In [21]:
from openai import OpenAI
import os 
from dotenv import load_dotenv
load_dotenv(override=True)

key=os.getenv('GROQ_API_KEY')
url = "https://api.groq.com/openai/v1"
myAi = OpenAI(api_key=key, base_url=url)
Model = "openai/gpt-oss-120b"

In [22]:
def messages_for(text, source_language, target_language):
    system_prompt = f"""
    You are a professional language translation assistant.
    Translate text from {source_language} to {target_language}.
    Preserve the original meaning, tone, formatting, and context.
    Respond only with the translated text.
    """

    user_prompt = f"""
    Translate the following text from {source_language}
    to {target_language}.

    Text:
    {text}

    Respond only with the translation.
    """

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [26]:
def translate(client,text,lang1,lang2):
    response  = client.chat.completions.create(
        model = Model,
        messages =  messages_for(text,lang1,lang2),
        reasoning_effort = "high"
    )
    reply = response.choices[0].message.content or ""
    # reply = reply.replace(f'```{lang2}', '').replace('```', '').strip()
    return reply

In [27]:
LANGUAGES = [
    "English",
    "Hindi",
    "Bengali",
    "Telugu",
    "Marathi",
    "Tamil",
    "Gujarati",
    "Urdu",
    "Kannada",
    "Odia",
    "Malayalam",
    "Punjabi",
    "Assamese",
    "Sanskrit",
    "Nepali",
    "French",
    "German",
    "Spanish",
    "Italian",
    "Portuguese",
    "Russian",
    "Japanese",
    "Korean",
    "Chinese",
    "Arabic",
    "Turkish"
]

In [28]:
import gradio as gr

def convert_lang(text,lang1, lang2):
    if not text or not text.strip():
        return "Please enter source code first."

    if lang1 == lang2:
        return text

    try:
        return translate(
            myAi,
            text,
            lang1,
            lang2
        )
    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks(title = "Translator") as demo:
    gr.Markdown("# 🔄 AI Translator")
    gr.Markdown(
        "Convert code from one language to another using AI."
    )

    with gr.Row():

        # LEFT SIDE - INPUT
        with gr.Column():

            input_language = gr.Dropdown(
                choices=LANGUAGES,
                value="English",
                label="Input Language"
            )

            text = gr.Textbox(
                label="text",
                placeholder="Paste your text here...",
                lines=20,
                max_lines=30
            )

        # RIGHT SIDE - OUTPUT
        with gr.Column():

            output_language = gr.Dropdown(
                choices=LANGUAGES,
                label="Target Language"
            )

            target_text = gr.Textbox(
                label="Converted text",
                placeholder="Converted text will appear here...",
                lines=20,
                max_lines=30
            )

    convert_button = gr.Button(
        "🚀 Convert Text",
        variant="secondary"
    )

    convert_button.click(
        fn=convert_lang,
        inputs=[
            text,
            input_language,
            output_language
        ],
        outputs=target_text
    )

    gr.Markdown(
        "**Tip:** Select the input and target languages, "
        "paste your code, then click Convert ."
    )


demo.launch(share=True, inline=False)


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://d4ca788b53ae8ce4a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
